## Character Text Splitter

In [1]:
text = '''Once upon a midnight dreary,
While I pondered, weak and weary,
Over many a quaint and curious
Volume of forgotten lore—
While I nodded, nearly napping,
Suddenly there came a tapping,
As of some one gently rapping,
Rapping at my chamber door.
"'T is some visitor," I muttered,
"Tapping at my chamber door
Only this and nothing more."

Ah, distinctly I remember,
It was in the bleak December,
And each separate dying ember
Wrought its ghost upon the floor.
Eagerly I wished the morrow;
Vainly I had sought to borrow
From my books surcease of sorrow
Sorrow for the lost Lenore—
For the rare and radiant maiden
Whom the angels name Lenore—
Nameless here for evermore.

And the silken, sad, uncertain
Rustling of each purple curtain
Thrilled me,—filled me with fantastic
Terrors, never felt before;
So that now, to still the beating
Of my heart, I stood repeating,
" 'T is some visitor entreating
Entrance at my chamber door
Some late visitor entreating
Entrance at my chamber door;
This it is and nothing more."'''

In [2]:
from langchain_community.document_loaders import TextLoader

loader = TextLoader("example.txt",encoding = 'utf-8')

text_loaded = loader.load()

e:\Langchain\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
from langchain_text_splitters import CharacterTextSplitter

text_splitter = CharacterTextSplitter(
separator=" ",
chunk_size=50,
chunk_overlap=5
)
texts = text_splitter.split_text(text)
len(texts)

24

In [4]:
print(texts)

['Once upon a midnight dreary,\nWhile I pondered,', 'weak and weary,\nOver many a quaint and', 'and curious\nVolume of forgotten lore—\nWhile I', 'I nodded, nearly napping,\nSuddenly there came a', 'a tapping,\nAs of some one gently rapping,\nRapping', 'at my chamber door.\n"\'T is some visitor," I', 'I muttered,\n"Tapping at my chamber door\nOnly this', 'this and nothing more."\n\nAh, distinctly I', 'I remember,\nIt was in the bleak December,\nAnd each', 'each separate dying ember\nWrought its ghost upon', 'upon the floor.\nEagerly I wished the', 'the morrow;\nVainly I had sought to borrow\nFrom my', 'my books surcease of sorrow\nSorrow for the lost', 'lost Lenore—\nFor the rare and radiant maiden\nWhom', 'the angels name Lenore—\nNameless here for', 'for evermore.\n\nAnd the silken, sad,', 'sad, uncertain\nRustling of each purple', 'curtain\nThrilled me,—filled me with', 'with fantastic\nTerrors, never felt before;\nSo that', 'that now, to still the beating\nOf my heart, I', 'I stood

In [5]:
chunks = text_splitter.split_documents(text_loaded)
len(chunks)

484

### Text Splitter structured Based

In [6]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=100,
    chunk_overlap=20,
    separators=["\n\n", "\n", " ", ""]
)

texts = splitter.split_text(text)
len(texts)

12

In [7]:
texts

['Once upon a midnight dreary,\nWhile I pondered, weak and weary,\nOver many a quaint and curious',
 'Volume of forgotten lore—\nWhile I nodded, nearly napping,\nSuddenly there came a tapping,',
 'As of some one gently rapping,\nRapping at my chamber door.\n"\'T is some visitor," I muttered,',
 '"Tapping at my chamber door\nOnly this and nothing more."',
 'Ah, distinctly I remember,\nIt was in the bleak December,\nAnd each separate dying ember',
 'Wrought its ghost upon the floor.\nEagerly I wished the morrow;\nVainly I had sought to borrow',
 'From my books surcease of sorrow\nSorrow for the lost Lenore—\nFor the rare and radiant maiden',
 'Whom the angels name Lenore—\nNameless here for evermore.',
 'And the silken, sad, uncertain\nRustling of each purple curtain',
 'Thrilled me,—filled me with fantastic\nTerrors, never felt before;\nSo that now, to still the beating',
 'Of my heart, I stood repeating,\n" \'T is some visitor entreating\nEntrance at my chamber door',
 'Some late visit

## Document Structured Based

In [8]:
pycode_text = '''from langchain_core.runnables import RunnableBranch,RunnableLambda
from pydantic import BaseModel, Field
from langchain_core.prompts import PromptTemplate
from langchain_groq import ChatGroq
from typing import Literal
from langchain_core.output_parsers import PydanticOutputParser, StrOutputParser

class SentimentResponse(BaseModel):
    sentiment: Literal["positive", "negative", "neutral"] = Field(
        description="The sentiment of the text")
    
prompt = PromptTemplate(
    template = "{instruction} give the sentiment of the following text: {text} "
)
model = ChatGroq( model = "openai/gpt-oss-20b")
parser = PydanticOutputParser(pydantic_object=SentimentResponse)



chain1 = prompt | model | parser | RunnableLambda(lambda x: {"sentiment": x.sentiment})
conditional_chain = RunnableBranch(
    (lambda x: x["sentiment"] == "positive",
        RunnableLambda(lambda x: f"The text is {x['sentiment']}. Great job!")),
    (lambda x: x["sentiment"] == "negative",
        RunnableLambda(lambda x: f"The text is {x['sentiment']}. Try to be more positive!")),
    (lambda x: x["sentiment"] == "neutral",
        RunnableLambda(lambda x: f"The text is {x['sentiment']}. It's balanced.")),
    RunnableLambda(lambda x: "Sentiment could not be determined.")
)

chain = chain1 | conditional_chain
'''

In [9]:
from langchain_text_splitters import RecursiveCharacterTextSplitter,Language
splitter = RecursiveCharacterTextSplitter.from_language(
    chunk_size=100,
    chunk_overlap=20,
    language= Language.PYTHON
)

chunks = splitter.split_text(pycode_text)
len(chunks)

20

In [10]:
chunks

['from langchain_core.runnables import RunnableBranch,RunnableLambda',
 'from pydantic import BaseModel, Field\nfrom langchain_core.prompts import PromptTemplate',
 'from langchain_groq import ChatGroq\nfrom typing import Literal',
 'from langchain_core.output_parsers import PydanticOutputParser, StrOutputParser',
 'class SentimentResponse(BaseModel):',
 'sentiment: Literal["positive", "negative", "neutral"] = Field(',
 'description="The sentiment of the text")',
 'prompt = PromptTemplate(',
 'template = "{instruction} give the sentiment of the following text: {text} "\n)',
 ')\nmodel = ChatGroq( model = "openai/gpt-oss-20b")',
 'parser = PydanticOutputParser(pydantic_object=SentimentResponse)',
 'chain1 = prompt | model | parser | RunnableLambda(lambda x: {"sentiment": x.sentiment})',
 'conditional_chain = RunnableBranch(\n    (lambda x: x["sentiment"] == "positive",',
 'RunnableLambda(lambda x: f"The text is {x[\'sentiment\']}. Great job!")),',
 '(lambda x: x["sentiment"] == "negativ

## sementic based Chunking

In [11]:
from langchain_experimental.text_splitter import SemanticChunker

# Vector Store

## FAISS

In [12]:
!pip install faiss-cpu 


[notice] A new release of pip available: 22.3 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [13]:
from langchain_community.document_loaders import TextLoader
loader = TextLoader("example.txt",encoding = 'utf-8')
text_loaded = loader.load()

In [14]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
splitter = RecursiveCharacterTextSplitter(chunk_size=200, chunk_overlap=10)
text = splitter.split_documents(text_loaded)

In [15]:
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

vector_store = FAISS.from_documents( #from_documents
    text,
    embedding=embeddings
)

In [16]:
res = vector_store.similarity_search("What are you going to do?", k = 3)
res

[Document(id='2aa9cf47-232d-4169-a6eb-3cc81367cb5f', metadata={'source': 'example.txt'}, page_content='"What are you going to do?" he whispered hoarsely.'),
 Document(id='30f25b58-1dec-402c-a10b-988dad0bc8f4', metadata={'source': 'example.txt'}, page_content='up to bed.'),
 Document(id='2dca2f95-09cb-4a6c-9211-e45befe3d188', metadata={'source': 'example.txt'}, page_content='make much out of it."')]

In [17]:
vector_store.add_texts(["This is a new document to add to the vector store."])

['e2bb230e-9dfc-452a-b50e-a2df8e680894']

In [18]:
vector_store.similarity_search_with_relevance_scores("What are you going to do?.", k=2)

[(Document(id='2aa9cf47-232d-4169-a6eb-3cc81367cb5f', metadata={'source': 'example.txt'}, page_content='"What are you going to do?" he whispered hoarsely.'),
  np.float32(0.1374017)),
 (Document(id='30f25b58-1dec-402c-a10b-988dad0bc8f4', metadata={'source': 'example.txt'}, page_content='up to bed.'),
  np.float32(0.06442642))]